# 09-2 — Algoritmo Genético para Selección de Características (GPU)

Misma implementación que `09_metodos_evolutivos.ipynb`, optimizada para ejecución en GPU.

| Cambio vs CPU | Valor CPU | Valor GPU |
|---|---|---|
| Mixed precision | — | `float16` (Tensor Cores) |
| Batch size fitness proxy | 256 | 512 |
| Epochs proxy | 20 | 25 |
| Tiempo estimado total | 1.5–2.5 h | **25–45 min** |

| Parámetro GA | Valor |
|---|---|
| Cromosoma | 15 bits |
| Población | 20 individuos |
| Generaciones máx | 15 · parada anticipada: 5 generaciones sin mejora |
| Selección | Torneo k=3 |
| Crossover | Un punto · prob=0.80 |
| Mutación | 2% por gen · mínimo 3 features activas |
| Elitismo | Top-1 por generación |
| Fitness proxy | 20% train · epochs=25 · EarlyStopping patience=2 |
| Caché | dict keyed por cromosoma |


In [ ]:
import os
import random
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    r2_score, f1_score, roc_auc_score,
    mean_absolute_error, precision_recall_curve
)

warnings.filterwarnings('ignore')

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── GPU: requerido ────────────────────────────────────────────────────────
gpus = tf.config.experimental.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError(
        '❌ No se detectó GPU. Ejecuta este notebook en el entorno tf_gpu '
        '(conda activate tf_gpu) o usa 09_metodos_evolutivos.ipynb para CPU.'
    )
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f'✓ GPU disponible: {len(gpus)} dispositivo(s)')
for g in gpus:
    print(f'  → {g.name}')

# ── Mixed precision float16 (Tensor Cores) ───────────────────────────────
mixed_precision.set_global_policy('mixed_float16')
print(f'✓ Política de precisión: {mixed_precision.global_policy().name}')
print(f'TensorFlow {tf.__version__}')

DATA_DIR     = Path('../dataset')
OUTPUTS_DIR  = Path('../outputs')
GRAFICAS_DIR = Path('../outputs/graficas')
MODELS_DIR   = Path('../models')
for d in [OUTPUTS_DIR, GRAFICAS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


RuntimeError: ❌ No se detectó GPU. Ejecuta este notebook en el entorno tf_gpu (conda activate tf_gpu) o usa 09_metodos_evolutivos.ipynb para CPU.

## 1. Carga de Datos y Preprocesamiento


In [ ]:
orders    = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'])
items     = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
products  = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
cat_names = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

df = orders[(orders['order_status'] == 'delivered') &
             orders['order_delivered_customer_date'].notna()].copy()
df['dias_entrega']   = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['es_retraso']     = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df = df[(df['dias_entrega'] >= 0) & (df['dias_entrega'] <= 60)]
df['dias_estimados'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days

df['mes_compra']        = df['order_purchase_timestamp'].dt.month
df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek
df['hora_compra']       = df['order_purchase_timestamp'].dt.hour

items_agg = items.sort_values('price', ascending=False).groupby('order_id').agg(
    precio_total=('price','sum'), flete_total=('freight_value','sum'),
    product_id=('product_id','first'), seller_id=('seller_id','first'),
    n_items=('order_item_id','count'), shipping_limit_date=('shipping_limit_date','first')
).reset_index()
df = df.merge(items_agg, on='order_id', how='left')
df['dias_limite_envio'] = (df['shipping_limit_date'] - df['order_purchase_timestamp']).dt.days.clip(lower=0)

products = products.merge(cat_names, on='product_category_name', how='left')
df = df.merge(products[['product_id','product_category_name_english',
    'product_weight_g','product_length_cm','product_height_cm','product_width_cm']],
    on='product_id', how='left')
df.rename(columns={'product_category_name_english': 'categoria_producto'}, inplace=True)
df['volumen_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

df = df.merge(sellers[['seller_id','seller_zip_code_prefix','seller_state']], on='seller_id', how='left')
df = df.merge(customers[['customer_id','customer_zip_code_prefix','customer_state']], on='customer_id', how='left')
df['mismo_estado'] = (df['seller_state'] == df['customer_state']).astype(int)

geo_agg = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat','median'), lng=('geolocation_lng','median')).reset_index()
df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix':'seller_zip_code_prefix',
    'lat':'seller_lat','lng':'seller_lng'}), on='seller_zip_code_prefix', how='left')
df = df.merge(geo_agg.rename(columns={'geolocation_zip_code_prefix':'customer_zip_code_prefix',
    'lat':'customer_lat','lng':'customer_lng'}), on='customer_zip_code_prefix', how='left')

def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lng2-lng1)/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distancia_km'] = haversine(df['seller_lat'], df['seller_lng'],
                                df['customer_lat'], df['customer_lng'])

print(f'Dataset: {len(df):,} órdenes | Retrasos: {df["es_retraso"].mean()*100:.1f}%')


In [ ]:
FEATURES_NUM = [
    'distancia_km', 'precio_total', 'flete_total',
    'product_weight_g', 'volumen_cm3',
    'mes_compra', 'dia_semana_compra', 'hora_compra',
    'dias_estimados', 'dias_limite_envio', 'n_items', 'mismo_estado'
]
FEATURES_CAT  = ['categoria_producto', 'customer_state', 'seller_state']
FEATURE_NAMES = FEATURES_NUM + FEATURES_CAT
N_FEATURES    = len(FEATURE_NAMES)   # 15

print(f'Features totales: {N_FEATURES}')
for i, f in enumerate(FEATURE_NAMES):
    tipo = 'NUM' if i < len(FEATURES_NUM) else 'CAT'
    print(f'  [{i:2d}] {tipo}  {f}')

# ── Split cronologico 80/10/10 ────────────────────────────────────────────
df_modelo = df[FEATURES_NUM + FEATURES_CAT + ['dias_entrega','es_retraso','order_purchase_timestamp']].copy()
for col in FEATURES_NUM:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())
for col in FEATURES_CAT:
    df_modelo[col] = df_modelo[col].fillna('desconocido')

df_modelo = df_modelo.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df_modelo)
n_train, n_val = int(n * 0.80), int(n * 0.90)
train = df_modelo.iloc[:n_train].copy()
val   = df_modelo.iloc[n_train:n_val].copy()
test  = df_modelo.iloc[n_val:].copy()

scaler = StandardScaler()
train[FEATURES_NUM] = scaler.fit_transform(train[FEATURES_NUM])
val[FEATURES_NUM]   = scaler.transform(val[FEATURES_NUM])
test[FEATURES_NUM]  = scaler.transform(test[FEATURES_NUM])

encoders, cardinalidades, IDX_DESCONOCIDO = {}, {}, {}
for col in FEATURES_CAT:
    le = LabelEncoder()
    # Forzar 'desconocido' en el fit para que siempre este en le.classes_
    train_vals = list(train[col].unique())
    if 'desconocido' not in train_vals:
        train_vals.append('desconocido')
    le.fit(sorted(train_vals))
    for split in [train, val, test]:
        split[col] = split[col].apply(lambda x: x if x in le.classes_ else 'desconocido')
    train[col] = le.transform(train[col])
    val[col]   = le.transform(val[col])
    test[col]  = le.transform(test[col])
    encoders[col]        = le
    cardinalidades[col]  = len(le.classes_)
    IDX_DESCONOCIDO[col] = int(le.transform(['desconocido'])[0])

rng      = np.random.RandomState(SEED)
sub_idx  = rng.choice(len(train), size=int(len(train) * 0.20), replace=False)
train_sub = train.iloc[sub_idx].reset_index(drop=True)

def preparar_inputs(df_split, mask_num=None, mask_cat=None):
    if mask_num is None: mask_num = np.ones(len(FEATURES_NUM), dtype=bool)
    if mask_cat is None: mask_cat = np.ones(len(FEATURES_CAT), dtype=bool)
    num_arr = df_split[FEATURES_NUM].values.astype('float32').copy()
    num_arr[:, ~mask_num.astype(bool)] = 0.0
    inputs = [num_arr]
    for i, col in enumerate(FEATURES_CAT):
        arr = df_split[col].values.astype('int32').copy()
        if not mask_cat[i]:
            arr[:] = IDX_DESCONOCIDO[col]
        inputs.append(arr)
    return inputs

y_train_reg     = train['dias_entrega'].values.astype('float32')
y_val_reg       = val['dias_entrega'].values.astype('float32')
y_train_clf     = train['es_retraso'].values.astype('float32')
y_val_clf       = val['es_retraso'].values.astype('float32')
y_train_sub_reg = train_sub['dias_entrega'].values.astype('float32')
y_train_sub_clf = train_sub['es_retraso'].values.astype('float32')

print(f'\nTrain: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'Sub-muestra fitness: {len(train_sub):,} filas (20%)')
print(f'Cardinalidades: {cardinalidades}')


## 2. Arquitectura de la Red Neuronal (fija)

> **GPU + Mixed precision**: la capa de salida usa `dtype='float32'` explícito para estabilidad numérica (las capas internas operan en `float16`).


In [ ]:
CAPAS   = [256, 128, 64]
DROPOUT = (0.3, 0.2)

def construir_modelo(capas, dropout, activacion_salida, nombre_salida):
    inp_num = keras.Input(shape=(len(FEATURES_NUM),), name='input_numerico')
    emb_inputs, emb_outputs = [], []
    for col, card in cardinalidades.items():
        dim = min(50, (card // 2) + 1)
        inp = keras.Input(shape=(1,), name=f'input_{col}', dtype='int32')
        emb = layers.Embedding(input_dim=card, output_dim=dim, name=f'emb_{col}')(inp)
        emb = layers.Flatten()(emb)
        emb_inputs.append(inp)
        emb_outputs.append(emb)

    x = layers.Concatenate()([inp_num] + emb_outputs)
    for i, u in enumerate(capas):
        x = layers.Dense(u, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout[0] if i == 0 else dropout[1])(x)

    # dtype='float32' en la salida: estabilidad numérica con mixed precision float16
    out = layers.Dense(1, activation=activacion_salida, name=nombre_salida,
                       dtype='float32')(x)
    return keras.Model(inputs=[inp_num] + emb_inputs, outputs=out)


def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        y_pred  = tf.cast(tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7), tf.float32)
        y_true  = tf.cast(y_true, tf.float32)
        bce     = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1 - p_t, gamma) * bce)
    return loss


def umbral_optimo(modelo, X_val, y_val):
    probs = modelo.predict(X_val, verbose=0).flatten()
    prec, rec, thr = precision_recall_curve(y_val, probs)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
    return float(thr[np.argmax(f1s)])

print('Arquitectura y funciones definidas.')
print(f'  Capas: {CAPAS}  Dropout: {DROPOUT}')
print(f'  Política activa: {mixed_precision.global_policy().name}')


## 3. Operadores Genéticos


In [ ]:
POP_SIZE   = 20
N_GEN      = 15
TOUR_K     = 3
CX_PROB    = 0.80
MUT_PROB   = 0.02
MIN_FEAT   = 3
PATIENCE   = 5

# GPU: batch mayor para mejor saturación de VRAM; +5 epochs porque es más rápido
FIT_EPOCHS   = 25
FIT_PATIENCE = 2
FIT_BATCH    = 512   # 256 en CPU → 512 en GPU


def cromosoma_a_mascaras(cromosoma):
    c = np.array(cromosoma, dtype=bool)
    return c[:len(FEATURES_NUM)], c[len(FEATURES_NUM):]


def reparar(cromosoma):
    c = list(cromosoma)
    if sum(c) < MIN_FEAT:
        inactivos = [i for i, v in enumerate(c) if v == 0]
        activar   = random.sample(inactivos, MIN_FEAT - sum(c))
        for i in activar:
            c[i] = 1
    return tuple(c)


def init_poblacion(n):
    poblacion = []
    for _ in range(n):
        c = tuple(random.randint(0, 1) for _ in range(N_FEATURES))
        poblacion.append(reparar(c))
    return poblacion


def torneo(poblacion, fitnesses, k=TOUR_K):
    candidatos = random.sample(range(len(poblacion)), k)
    mejor = max(candidatos, key=lambda i: fitnesses[i])
    return poblacion[mejor]


def cruce_un_punto(p1, p2, prob=CX_PROB):
    if random.random() < prob:
        punto = random.randint(1, N_FEATURES - 1)
        h1 = reparar(p1[:punto] + p2[punto:])
        h2 = reparar(p2[:punto] + p1[punto:])
        return h1, h2
    return reparar(p1), reparar(p2)


def mutar(cromosoma, prob=MUT_PROB):
    c = list(cromosoma)
    for i in range(len(c)):
        if random.random() < prob:
            c[i] = 1 - c[i]
    return reparar(tuple(c))


print('Operadores genéticos definidos.')
print(f'  POP={POP_SIZE} | N_GEN={N_GEN} | FIT_BATCH={FIT_BATCH} | FIT_EPOCHS={FIT_EPOCHS}')


## 4. Función de Fitness (proxy con submuestra)


In [ ]:
def evaluar_fitness(cromosoma, tarea, cache):
    key = (tarea,) + cromosoma
    if key in cache:
        return cache[key]

    mask_num, mask_cat = cromosoma_a_mascaras(cromosoma)
    X_sub = preparar_inputs(train_sub, mask_num, mask_cat)
    X_v   = preparar_inputs(val,       mask_num, mask_cat)

    keras.backend.clear_session()

    if tarea == 'reg':
        y_sub, y_v = y_train_sub_reg, y_val_reg
        modelo = construir_modelo(CAPAS, DROPOUT, 'linear', 'dias_entrega')
        modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mae', metrics=['mae'])
        monitor, mode = 'val_loss', 'min'
    else:
        y_sub, y_v = y_train_sub_clf, y_val_clf
        modelo = construir_modelo(CAPAS, DROPOUT, 'sigmoid', 'es_retraso')
        modelo.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss=focal_loss(gamma=2.0, alpha=0.75),
            metrics=[keras.metrics.AUC(name='auc')]
        )
        monitor, mode = 'val_auc', 'max'

    cb = [keras.callbacks.EarlyStopping(
              monitor=monitor, patience=FIT_PATIENCE,
              restore_best_weights=True, mode=mode, verbose=0),
          keras.callbacks.ReduceLROnPlateau(
              monitor='val_loss', factor=0.5, patience=2, verbose=0)]

    modelo.fit(X_sub, y_sub,
               validation_data=(X_v, y_v),
               epochs=FIT_EPOCHS, batch_size=FIT_BATCH,
               callbacks=cb, verbose=0)

    if tarea == 'reg':
        pred    = modelo.predict(X_v, verbose=0).flatten()
        fitness = float(r2_score(y_v, pred))
    else:
        probs   = modelo.predict(X_v, verbose=0).flatten()
        thr     = umbral_optimo(modelo, X_v, y_v)
        preds   = (probs >= thr).astype(int)
        fitness = float(f1_score(y_v, preds, zero_division=0))

    del modelo
    cache[key] = fitness
    return fitness


def evaluar_poblacion(poblacion, tarea, cache):
    return [evaluar_fitness(c, tarea, cache) for c in poblacion]


print('Función de fitness definida (GPU).')
print(f'  Proxy: {len(train_sub):,} filas | batch={FIT_BATCH} | epochs={FIT_EPOCHS} | patience={FIT_PATIENCE}')


## 5. Loop Principal del GA


In [ ]:
def run_ga(tarea):
    label    = 'GA-REG (R²)' if tarea == 'reg' else 'GA-CLF (F1)'
    cache    = {}
    historia = []

    poblacion = init_poblacion(POP_SIZE)
    fitnesses = evaluar_poblacion(poblacion, tarea, cache)

    mejor_global = max(fitnesses)
    elite_global = poblacion[fitnesses.index(mejor_global)]
    sin_mejora   = 0

    print(f'\n{"="*55}')
    print(f'  {label}  — {N_GEN} generaciones máx · parada: {PATIENCE} sin mejora')
    print(f'{"="*55}')
    print(f'  {"Gen":>3}  {"Mejor":>8}  {"Promedio":>8}  {"Sin mejora":>10}  Features activas')
    print(f'  {"-"*55}')

    for gen in range(N_GEN):
        elite_idx = fitnesses.index(max(fitnesses))
        elite     = poblacion[elite_idx]

        nueva_gen = [elite]
        while len(nueva_gen) < POP_SIZE:
            p1 = torneo(poblacion, fitnesses)
            p2 = torneo(poblacion, fitnesses)
            h1, h2 = cruce_un_punto(p1, p2)
            nueva_gen.append(mutar(h1))
            if len(nueva_gen) < POP_SIZE:
                nueva_gen.append(mutar(h2))

        poblacion = nueva_gen
        fitnesses = evaluar_poblacion(poblacion, tarea, cache)

        mejor_gen = max(fitnesses)
        avg_gen   = np.mean(fitnesses)
        elite_gen = poblacion[fitnesses.index(mejor_gen)]
        n_activas = sum(elite_gen)

        historia.append({
            'gen':           gen + 1,
            'mejor_fitness': round(mejor_gen, 6),
            'avg_fitness':   round(float(avg_gen), 6),
            'elite':         list(elite_gen),
        })
        print(f'  {gen+1:>3}  {mejor_gen:>8.4f}  {avg_gen:>8.4f}  {sin_mejora:>10}  {n_activas}/{N_FEATURES}')

        if mejor_gen > mejor_global:
            mejor_global = mejor_gen
            elite_global = elite_gen
            sin_mejora   = 0
        else:
            sin_mejora += 1

        if sin_mejora >= PATIENCE:
            print(f'\n  ⚑ Parada anticipada: {PATIENCE} generaciones sin mejora.')
            break

    print(f'\n  Mejor {label}: {mejor_global:.4f}')
    print(f'  Features seleccionadas ({sum(elite_global)}/{N_FEATURES}):')
    for i, (bit, nombre) in enumerate(zip(elite_global, FEATURE_NAMES)):
        print(f'    {"✓" if bit else "✗"} [{i:2d}] {nombre}')

    return list(elite_global), historia


print('Loop GA definido. Listo para ejecutar en GPU.')


## 6. Ejecución — GA Regresión (maximiza R²)


In [ ]:
cromosoma_reg, historia_reg = run_ga('reg')


## 7. Ejecución — GA Clasificación (maximiza F1)


In [ ]:
cromosoma_clf, historia_clf = run_ga('clf')


## 8. Visualizaciones de la Evolución


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evolución del Fitness por Generación (GPU)', fontsize=13, fontweight='bold')

for ax, historia, titulo, ylabel, color in [
    (axes[0], historia_reg, 'GA-REG — R² en validación',        'R²',       'steelblue'),
    (axes[1], historia_clf, 'GA-CLF — F1-score en validación',  'F1-score', 'seagreen'),
]:
    gens      = [h['gen']           for h in historia]
    mejores   = [h['mejor_fitness'] for h in historia]
    promedios = [h['avg_fitness']   for h in historia]

    ax.plot(gens, mejores,   'o-',  color=color, linewidth=2, markersize=6, label='Mejor')
    ax.fill_between(gens, mejores, promedios, alpha=0.15, color=color)
    ax.plot(gens, promedios, 's--', color=color, linewidth=1, markersize=4, alpha=0.7, label='Promedio')
    ax.set_xlabel('Generación'); ax.set_ylabel(ylabel)
    ax.set_title(titulo); ax.legend(); ax.grid(alpha=0.3)
    ax.set_xticks(gens)

plt.tight_layout()
plt.savefig(GRAFICAS_DIR / 'ga_evolucion_fitness_gpu.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: outputs/graficas/ga_evolucion_fitness_gpu.png')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Features Seleccionadas por el GA (GPU)', fontsize=13, fontweight='bold')

for ax, cromosoma, titulo, color_on, color_off in [
    (axes[0], cromosoma_reg, f'GA-REG  ({sum(cromosoma_reg)}/{N_FEATURES} features)', '#2196F3', '#ECEFF1'),
    (axes[1], cromosoma_clf, f'GA-CLF  ({sum(cromosoma_clf)}/{N_FEATURES} features)', '#4CAF50', '#ECEFF1'),
]:
    y_pos   = range(N_FEATURES - 1, -1, -1)
    colores = [color_on if b else color_off for b in cromosoma]
    ax.barh(list(y_pos), [1] * N_FEATURES, color=colores, edgecolor='white', linewidth=1.5)

    for yp, bit, nombre in zip(y_pos, cromosoma, FEATURE_NAMES):
        ax.text(0.5, yp, nombre, va='center', ha='center', fontsize=9,
                color='white' if bit else '#666666', fontweight='bold' if bit else 'normal')
        ax.text(1.05, yp, '✓ INCLUIDA' if bit else '✗ excluida', va='center', fontsize=8,
                color=color_on if bit else '#AAAAAA')

    tipo_labels = ['NUM'] * len(FEATURES_NUM) + ['CAT'] * len(FEATURES_CAT)
    for yp, tipo in zip(y_pos, tipo_labels):
        ax.text(-0.05, yp, tipo, va='center', ha='right', fontsize=7, color='#888888')

    ax.set_xlim(-0.15, 1.7); ax.set_yticks([]); ax.set_xticks([])
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.spines[['top','right','left','bottom']].set_visible(False)

    inc_patch = mpatches.Patch(color=color_on,  label='Incluida')
    exc_patch = mpatches.Patch(color=color_off, label='Excluida', edgecolor='#AAAAAA')
    ax.legend(handles=[inc_patch, exc_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(GRAFICAS_DIR / 'ga_features_seleccionadas_gpu.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: outputs/graficas/ga_features_seleccionadas_gpu.png')


## 9. Guardar Cromosomas Óptimos


In [ ]:
features_reg = [FEATURE_NAMES[i] for i, b in enumerate(cromosoma_reg) if b]
features_clf = [FEATURE_NAMES[i] for i, b in enumerate(cromosoma_clf) if b]

ga_resultados = {
    "feature_names": FEATURE_NAMES,
    "ga_regresion": {
        "cromosoma":        cromosoma_reg,
        "features_activas": features_reg,
        "n_features":       sum(cromosoma_reg),
        "mejor_fitness_r2": historia_reg[-1]['mejor_fitness'],
        "historia":         historia_reg,
    },
    "ga_clasificacion": {
        "cromosoma":        cromosoma_clf,
        "features_activas": features_clf,
        "n_features":       sum(cromosoma_clf),
        "mejor_fitness_f1": historia_clf[-1]['mejor_fitness'],
        "historia":         historia_clf,
    },
    "parametros_ga": {
        "pop_size":    POP_SIZE,
        "n_gen":       N_GEN,
        "tour_k":      TOUR_K,
        "cx_prob":     CX_PROB,
        "mut_prob":    MUT_PROB,
        "min_feat":    MIN_FEAT,
        "patience":    PATIENCE,
        "fit_epochs":  FIT_EPOCHS,
        "fit_batch":   FIT_BATCH,
        "subsample":   len(train_sub),
        "precision":   mixed_precision.global_policy().name,
    }
}

with open(OUTPUTS_DIR / 'ga_resultados.json', 'w', encoding='utf-8') as f:
    json.dump(ga_resultados, f, ensure_ascii=False, indent=2)

print('Guardado: outputs/ga_resultados.json')
print(f'\nGA-REG  → {sum(cromosoma_reg)}/{N_FEATURES} features: {features_reg}')
print(f'GA-CLF  → {sum(cromosoma_clf)}/{N_FEATURES} features: {features_clf}')
